# Lesson 07
Run lessons 1–6 first, or use 00_run_all.ipynb. This is the actual teaching source, not a placeholder.

# 第7课 市场微观结构与执行实验

目标：从LOBSTER message/orderbook理解价差、深度、不平衡、microprice、订单流、库存与逆向选择。microprice=(ask×bid_size+bid×ask_size)/(bid_size+ask_size)是报价代理，不是保证可成交价格。

优先读取官方2012-06-21 AAPL小样例；若官方端点不可用，严格标记为合成事件，只验证机制。不能把合成/单日结果写成真实高频盈利证据，也不套用252年化逐笔Sharpe。原始受限数据不打包进GitHub。

实验A：TWAP vs 使用更早时段成交量形状估计的VWAP调度；不得用执行窗实际成交量设计“事前VWAP”。买单成交使用延迟后可见卖一档，限制参与和可见深度，未完成部分按期末可见深度强制完成并记入剩余/费用；实现差额使用决策到达mid作为基准，报告执行金额、未完成量和冲击情景。

实验B：双边报价加入库存偏移和限额；新单及撤改单均受延迟，排队前量按报价时可见同价深度估计。仅显示的反向主动成交在相同价位消耗队列后才可能成交；取消不自动改善队列；触价本身不算成交。不处理隐藏单，不以未来成交挑报价。

这是小订单市场回放，自己的订单不改变其他人的事件。延迟、队列倍数和费用做情景敏感性，模拟成本不能直接校准日频ETF/个股账户，除非市场、年代和粒度可比。

参考：[LOBSTER样例入口](https://lobsterdata.com/info/DataSamples.php)、[HftBacktest成交与队列假设](https://hftbacktest.readthedocs.io/en/latest/order_fill.html)。


In [ ]:
# 07.01 官方LOBSTER样例、消息解析及明确标记的合成回退
import requests,zipfile,re
LOB_URL='https://data.lobsterdata.com/info/sample/LOBSTER_SampleFile_AAPL_2012-06-21_1.zip'
LOB_MANIFEST={'requested_url':LOB_URL,'retrieved_utc':pd.Timestamp.now(tz='UTC').isoformat(),'redistribution':'raw data excluded from portfolio package'}
try:
    content=fetch(LOB_URL);archive=zipfile.ZipFile(io.BytesIO(content));names=archive.namelist()
    message_file=next(n for n in names if '_message_' in n and n.endswith('.csv'))
    book_file=next(n for n in names if '_orderbook_' in n and n.endswith('.csv'))
    MSG=pd.read_csv(io.BytesIO(archive.read(message_file)),header=None,names=['time','type','order_id','size','price','direction'])
    BOOK=pd.read_csv(io.BytesIO(archive.read(book_file)),header=None).iloc[:,:4];BOOK.columns=['ask','ask_size','bid','bid_size']
    assert len(MSG)==len(BOOK)
    BOOK[['ask','bid']]=BOOK[['ask','bid']]/10000.;MSG['price']=MSG.price/10000.
    LOB_MANIFEST.update(source_kind='official_single_day_sample',sample_date='2012-06-21',sha256=hashlib.sha256(content).hexdigest(),rows=len(MSG))
except Exception as e:
    rng=np.random.default_rng(7001);n=50000;clock=np.linspace(34200,57600,n)
    ticks=10000+np.cumsum(rng.choice([-1,0,1],size=n,p=[.12,.76,.12]));mid=ticks*.01
    bid=mid-.01;ask=mid+.01;direction=rng.choice([-1,1],n);types=rng.choice([1,2,3,4],n,p=[.45,.12,.12,.31]);sizes=rng.integers(1,101,n)
    MSG=pd.DataFrame({'time':clock,'type':types,'order_id':np.arange(n),'size':sizes,'price':np.where(direction==1,bid,ask),'direction':direction})
    BOOK=pd.DataFrame({'ask':ask,'ask_size':rng.integers(50,501,n),'bid':bid,'bid_size':rng.integers(50,501,n)})
    LOB_MANIFEST.update(source_kind='SYNTHETIC_MECHANISM_ONLY',reason=str(e),seed=7001,rows=n,
      limitations='synthetic L1 quotes and event stream, not a full exchange order-ID reconstruction')
LOB=pd.concat([MSG,BOOK],axis=1)
LOB=LOB[(LOB.time>=34200)&(LOB.time<=57600)&(LOB.ask>LOB.bid)&(LOB.bid>0)&(LOB.ask_size>0)&(LOB.bid_size>0)].reset_index(drop=True)
LOB['mid']=(LOB.ask+LOB.bid)/2;LOB['spread']=LOB.ask-LOB.bid
LOB['imbalance']=(LOB.bid_size-LOB.ask_size)/(LOB.bid_size+LOB.ask_size)
LOB['microprice']=(LOB.ask*LOB.bid_size+LOB.bid*LOB.ask_size)/(LOB.bid_size+LOB.ask_size)
LOB['signed_trade_volume']=np.where(LOB.type.eq(4),-LOB.direction*LOB['size'],0)
LOB['past_order_flow']=LOB.signed_trade_volume.rolling(100,min_periods=1).sum()
LOB['past_mid_change']=LOB.mid.diff().fillna(0)
assert LOB.time.is_monotonic_increasing and np.isfinite(LOB[['mid','spread','imbalance','microprice']]).all().all()
print(json.dumps(LOB_MANIFEST,ensure_ascii=False,indent=2));display(LOB.head());display(LOB[['spread','imbalance','ask_size','bid_size']].describe())
fig,ax=plt.subplots(1,2,figsize=(12,4));LOB.iloc[::100].plot(x='time',y=['mid','microprice'],ax=ax[0],title=LOB_MANIFEST['source_kind'])
LOB.imbalance.hist(bins=40,ax=ax[1]);ax[1].set_title('L1 queue imbalance');plt.tight_layout();plt.show()
print('07.01 PASS: provenance and message/orderbook row mapping preserved')


In [ ]:
# 07.02 TWAP/事前VWAP调度与实现差额账本
CAL_START,CAL_END,EX_START,EX_END=34200.,39600.,43200.,48600.
CAL=LOB[(LOB.time>=CAL_START)&(LOB.time<CAL_END)&LOB.type.eq(4)].copy()
N_CHILD=9;Q_TARGET=1000;cal_bins=np.linspace(CAL_START,CAL_END,N_CHILD+1)
cal_volume=np.histogram(CAL.time,bins=cal_bins,weights=CAL['size'])[0].astype(float)
VWAP_PROFILE=(cal_volume+1)/(cal_volume.sum()+N_CHILD)
TWAP_PROFILE=np.ones(N_CHILD)/N_CHILD
assert CAL.time.max()<EX_START
EXEC7={};EXEC_REPORT7=[]
def execute_schedule(profile,latency=.25,participation=.1,impact_bps=0.):
    child_times=np.linspace(EX_START,EX_END,N_CHILD,endpoint=False)
    allocation=np.floor(Q_TARGET*np.asarray(profile)).astype(int);allocation[-1]+=Q_TARGET-allocation.sum()
    arrival_idx=max(0,LOB.time.searchsorted(EX_START,side='right')-1);arrival_mid=float(LOB.mid.iloc[arrival_idx]);used={};rows=[];remaining=Q_TARGET
    for t,q in zip(np.r_[child_times,EX_END],np.r_[allocation,Q_TARGET]):
        if remaining<=0:break
        i=int(LOB.time.searchsorted(t+latency,side='left'))
        if i>=len(LOB):break
        quote=LOB.iloc[i];available=max(0,int(quote.ask_size)-used.get(i,0));request=min(remaining,int(q))
        # 小订单显示深度限制；自身成交不改变回放市场，单事件重复使用深度被扣除。
        fill=min(request,int(available*participation));used[i]=used.get(i,0)+fill
        price=float(quote.ask)*(1+impact_bps/10000);fee=fill*price*.0001
        rows.append({'request_at':t,'ready_at':t+latency,'fill_at':quote.time,'event_index':i,'requested':request,'filled':fill,'price':price,'fee':fee,'remaining_after':remaining-fill,'forced_end':t==EX_END})
        remaining-=fill
    trades=pd.DataFrame(rows);filled=trades.filled.sum();notional=(trades.filled*trades.price).sum();fees=trades.fee.sum()
    end_idx=min(len(LOB)-1,int(LOB.time.searchsorted(EX_END+latency,side='left')));end_mid=float(LOB.mid.iloc[end_idx])
    report={'target':Q_TARGET,'filled':int(filled),'unfilled':int(remaining),'fill_rate':filled/Q_TARGET,'arrival_mid':arrival_mid,
      'VWAP_execution':notional/max(filled,1),'fees':fees,'IS_filled_bps':((notional+fees)/max(filled,1)/arrival_mid-1)*10000 if filled else np.nan,
      'opportunity_adjusted_IS_bps':((notional+fees+remaining*end_mid)/(Q_TARGET*arrival_mid)-1)*10000,
      'remaining_mark_is_not_a_fill':True,'latency_seconds':latency,'participation_of_displayed_depth':participation,'impact_assumption_bps':impact_bps}
    return trades,report
for name,profile in [('TWAP',TWAP_PROFILE),('past_volume_schedule',VWAP_PROFILE)]:
    for latency in [0.,.25,1.]:
        trades,report=execute_schedule(profile,latency=latency);key=name+'_latency_'+str(latency);EXEC7[key]=trades;EXEC_REPORT7.append({'algorithm':name,**report})
EXEC_RESULT7=pd.DataFrame(EXEC_REPORT7);display(pd.DataFrame({'TWAP':TWAP_PROFILE,'past_volume':VWAP_PROFILE,'calibration_volume':cal_volume}));display(EXEC_RESULT7)
print('事后市场VWAP只作描述，不回填事前调度。剩余订单按期末mid计机会成本，不能冒充真实成交。')
executed_window=LOB[(LOB.time>=EX_START)&(LOB.time<=EX_END)&LOB.type.eq(4)]
print('Ex-post market trade VWAP',np.average(executed_window.price,weights=executed_window['size']))
fig,ax=plt.subplots(1,2,figsize=(12,4));pd.DataFrame({'TWAP':TWAP_PROFILE,'past_volume':VWAP_PROFILE}).plot.bar(ax=ax[0],title='Schedules fixed before execution window')
for name,t in EXEC7.items():ax[1].step(t.fill_at,t.filled.cumsum(),where='post',label=name)
ax[1].axhline(Q_TARGET,ls='--',color='k');ax[1].legend(fontsize=7);ax[1].set_title('Cumulative actual fills');plt.tight_layout();plt.show()
print('07.02 PASS: no future volume used, displayed depth and incomplete fills retained')


In [ ]:
# 07.03 队列、撤改单延迟与库存控制做市原型
MM_WINDOW=LOB[(LOB.time>=EX_START)&(LOB.time<=EX_END)].copy().reset_index(drop=True)
TICK7=.01
# 消息type4为显示限价单执行；direction是被执行的挂单方向，不能当主动方方向。
def market_make(latency=.25,queue_multiplier=1.,inventory_control=True,fee_bps=1.):
    cash=0.;inv=0;orders={};pending=None;records=[];fills=[];limit=20;lot=5;last_refresh=-np.inf;requests=0
    for i in range(1,len(MM_WINDOW)):
        event=MM_WINDOW.iloc[i];prev=MM_WINDOW.iloc[i-1];now=float(event.time)
        if now-last_refresh>=10 and pending is None:
            shift=int(np.round(inv/5)) if inventory_control else 0
            desired={1:float(prev.bid)-shift*TICK7,-1:float(prev.ask)-shift*TICK7}
            pending={'ready':float(prev.time)+latency,'quotes':desired,'request_at':float(prev.time)};last_refresh=now;requests+=2
        if pending is not None and pending['ready']<=now:
            orders={} # cancellation only now effective; old quotes remained live through delay
            for side,price in pending['quotes'].items():
                best=float(prev.bid if side==1 else prev.ask);depth=float(prev.bid_size if side==1 else prev.ask_size)
                passive=(price<float(prev.ask) if side==1 else price>float(prev.bid))
                allowed=(inv+lot<=limit if side==1 else inv-lot>=-limit)
                if passive and allowed:
                    ahead=depth*queue_multiplier if abs(price-best)<TICK7/4 else np.inf
                    orders[side]={'price':price,'remaining':lot,'ahead':ahead,'activated_at':now,'request_at':pending['request_at']}
            pending=None
        if int(event['type'])==4:
            side=int(event.direction);order=orders.get(side)
            if order is not None and abs(float(event.price)-order['price'])<TICK7/4:
                before=order['ahead'];quantity=float(event['size']);order['ahead']=max(0,before-quantity)
                possible=max(0,quantity-before);q=int(min(order['remaining'],possible))
                q=min(q,limit-inv if side==1 else limit+inv)
                if q>0:
                    fee=q*order['price']*fee_bps/10000;cash-=side*q*order['price']+fee;inv+=side*q;order['remaining']-=q
                    fills.append({'time':now,'side':side,'quantity':q,'price':order['price'],'fee':fee,'inventory':inv,'ahead_before':before,
                       'request_at':order['request_at'],'activated_at':order['activated_at'],'source_type':4,'trade_quantity':quantity})
                if order['remaining']==0:orders.pop(side,None)
        assert abs(inv)<=limit
        records.append({'time':now,'cash':cash,'inventory':inv,'mid':float(event.mid),'equity':cash+inv*float(event.mid),'live_orders':len(orders)})
    path=pd.DataFrame(records);trades=pd.DataFrame(fills)
    # 在可见最优档容量内平仓；不足时只mark剩余，不捏造成交。
    last=MM_WINDOW.iloc[-1];closing=min(abs(inv),int(last.bid_size if inv>0 else last.ask_size))
    liquidation_price=float(last.bid if inv>0 else last.ask);side=-np.sign(inv);liq_fee=closing*liquidation_price*fee_bps/10000
    cash-=side*closing*liquidation_price+liq_fee;inv+=int(side*closing)
    final=cash+inv*float(last.mid)
    report={'latency_seconds':latency,'queue_multiplier':queue_multiplier,'inventory_control':inventory_control,'fee_bps':fee_bps,
      'source':LOB_MANIFEST['source_kind'],'posted_requests':requests,'filled_units':int(trades.quantity.sum()) if len(trades) else 0,
      'fill_events':len(trades),'net_marked_PnL':final,'max_abs_inventory':path.inventory.abs().max(),'remaining_inventory':inv,
      'closing_units':closing,'closing_fee':liq_fee,'min_marked_PnL':path.equity.min(),'source_window_events':len(path)}
    if len(trades):
        future_mid=[]
        for _,r in trades.iterrows():
            j=min(len(MM_WINDOW)-1,int(MM_WINDOW.time.searchsorted(r.time+5)))
            future_mid.append(r.side*(float(MM_WINDOW.mid.iloc[j])-r.price))
        trades['signed_5sec_markout']=future_mid;report['mean_markout_per_share']=np.average(trades.signed_5sec_markout,weights=trades.quantity)
    return path,trades,report
MM_PATHS7={};MM_FILLS7={};MM_REPORT=[]
for controlled in [False,True]:
    for latency in [0.,.25,1.]:
        for queue in [1.,2.]:
            key=(controlled,latency,queue);path,trades,report=market_make(latency,queue,controlled)
            MM_PATHS7[key]=path;MM_FILLS7[key]=trades;MM_REPORT.append(report)
MM_RESULT7=pd.DataFrame(MM_REPORT);display(MM_RESULT7)
fig,ax=plt.subplots(1,2,figsize=(13,4))
for key,p in MM_PATHS7.items():
    if key[2]==1:ax[0].plot(p.time,p.equity,label=str(key));ax[1].plot(p.time,p.inventory,label=str(key))
ax[0].legend(fontsize=7);ax[0].set_title('Marked PnL, replay queue/latency assumptions');ax[1].set_title('Inventory, hard cap ±20 shares');plt.tight_layout();plt.show()
print('取消不推进队列；触价不成交。改单就重新排队。未建模隐藏量、跨档优先级、冲击反馈与交易所微秒协议。')
L7_CHECKS=[]
def check7(name,ok):
    assert bool(ok),name;L7_CHECKS.append({'check':name,'status':'PASS'})
check7('source explicitly actual or synthetic',LOB_MANIFEST['source_kind'] in ['official_single_day_sample','SYNTHETIC_MECHANISM_ONLY'])
check7('volume schedule ends before execution',CAL.time.max()<EX_START)
check7('buy/sell quotes ordered',(LOB.ask>LOB.bid).all())
check7('microprice inside spread',((LOB.microprice>=LOB.bid-1e-9)&(LOB.microprice<=LOB.ask+1e-9)).all())
check7('imbalance in [-1,1]',LOB.imbalance.abs().max()<=1)
# 手算队列：前方100，第一笔60不能成交，第二笔50只能成交10。
ahead=100;fill1=max(0,60-ahead);ahead=max(0,ahead-60);fill2=min(20,max(0,50-ahead))
check7('manual queue conservation',fill1==0 and fill2==10)
for name,t in EXEC7.items():
    check7(name+' temporal latency',(t.fill_at>=t.ready_at).all())
    check7(name+' fill conservation',int(t.filled.sum()+t.remaining_after.iloc[-1])==Q_TARGET)
    check7(name+' partial not fabricated',(t.filled<=t.requested).all() and (t.fee>=0).all())
for key,p in MM_PATHS7.items():
    check7(str(key)+' cash-inventory mark',np.allclose(p.equity,p.cash+p.inventory*p.mid))
    check7(str(key)+' inventory cap',p.inventory.abs().max()<=20)
    f=MM_FILLS7[key]
    check7(str(key)+' displayed trade only',f.empty or f.source_type.eq(4).all())
    check7(str(key)+' latency and queue',f.empty or ((f.time>=f.activated_at)&(f.activated_at>=f.request_at+key[1])&(f.quantity<=np.maximum(0,f.trade_quantity-f.ahead_before)+1e-8)).all())
L7_VALIDATION=pd.DataFrame(L7_CHECKS);display(L7_VALIDATION)
L7_CARD={'lesson':7,'checks':len(L7_CHECKS),'source':LOB_MANIFEST,'status':'verified_mechanisms','limits':['single-day or synthetic explicitly marked','no market impact response','cancel/hidden queues uncertain','not evidence of live HFT profitability','not used to calibrate daily stock costs']}
print('L7 FINAL PASS',json.dumps(L7_CARD,ensure_ascii=False))
